# Peramalan Tiket Bioskop D4-D10 untuk Film Baru

JOINTS 2026 Data Science Competition. Notebook ini membawa pembaca dari data mentah sampai berkas submission,
dengan satu alur yang bisa dijalankan ulang dari sel pertama ke sel terakhir.

**Hasil**

| | Skor (MASE, lebih kecil lebih baik) |
|---|---|
| Titik awal: median rasio per horizon dan tersil skala | 0,573 |
| Submission akhir (`submission_juri_final.csv`) | **0,475** |

**Pesan utama**

1. Skor 0,573 ke 0,55 datang dari satu koreksi definisi: D1 di data latih harus **hari rilis lebar**, bukan tanggal transaksi pertama.
   Sebagian film punya hari premiere di satu klaster sebelum rilis, hari yang tidak pernah muncul di data uji.
2. Metrik adalah galat absolut dibagi skala, jadi prediksi yang tepat adalah **median**, bukan rata-rata.
   Model dilatih dengan loss L1 pada rasio terhadap skala.
3. Data latih (Apr-Sep 2025) dan data uji (Okt 2025-Mar 2026) berbeda **musim**: libur sekolah Desember, Natal, Imlek, Ramadan, dan Lebaran
   tidak ada padanan penuhnya di data latih. Dua koreksi musim yang punya padanan terukur di data latih (libur sekolah dan Lebaran)
   menurunkan skor dari 0,486 ke 0,475.
4. Semua sumber eksternal diaudit terhadap aturan lomba (tersedia publik paling lambat 30 September 2025). Sumber yang melanggar aturan
   dipakai hanya sebagai uji coba dan **tidak masuk model akhir**.

**Cara menjalankan.** Jalankan semua sel berurutan dengan kernel Python 3.11 yang punya `xgboost` (waktu eksekusi sekitar 2 menit).
Notebook membaca `data/*.csv`, mengunduh revisi Wikipedia tetap bila belum ada di `data/external`, dan menulis `data/submission_juri_final.csv`.
Semua sel acak memakai seed tetap, sehingga berkas yang dihasilkan sama setiap kali dijalankan.

**Isi**

0. Setup
1. Data dan audit kualitas
2. Membongkar aturan pembentukan data uji
3. Training set semu dan sifat target
4. Pergeseran data latih ke data uji
5. Protokol validasi
6. Rekayasa fitur
7. Model
8. Hasil validasi
9. Koreksi musim
10. Submission
11. Audit sumber eksternal, eksperimen yang gugur, dan batas

## 0. Setup

In [ ]:
import warnings, re, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.ensemble import HistGradientBoostingClassifier as HGC

try:
    import xgboost as xgb
except ImportError as e:
    raise ImportError('xgboost diperlukan: pip install xgboost, atau pilih kernel Python 3.11.') from e

warnings.filterwarnings('ignore')
DATA = next(p for p in (Path('dataset'), Path('data'), Path('../dataset'), Path('../data')) if (p / 'train.csv').exists())
EXT = DATA / 'external'
EXT.mkdir(exist_ok=True)

FMT = r'\s*\((IMAX|3D|2D|4DX|SCREENX)[^)]*\)\s*$'     # penanda format di akhir judul
judul_dasar = lambda t: re.sub(FMT, '', str(t)).strip()

C_TRAIN, C_TEST, C_ACC = '#2563eb', '#dc2626', '#16a34a'
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3, 'axes.titleweight': 'bold',
                     'axes.titlesize': 11, 'font.size': 10})

import sklearn
print('python-paket: numpy %s | pandas %s | sklearn %s | xgboost %s' % (np.__version__, pd.__version__, sklearn.__version__, xgb.__version__))
print('folder data :', DATA)

## 1. Data dan audit kualitas

Tujuh berkas resmi dari panitia. Tidak ada data lain yang dipakai sebagai input model selain yang didaftar di bagian 11.

In [ ]:
train = pd.read_csv(DATA / 'train.csv', parse_dates=['date_show'])
th = pd.read_csv(DATA / 'test_history.csv', parse_dates=['date_show'])
te = pd.read_csv(DATA / 'test.csv', parse_dates=['date_show'])
movies = pd.read_csv(DATA / 'movies.csv')
hol = pd.read_csv(DATA / 'holidays.csv', parse_dates=['date'])
prices = pd.read_csv(DATA / 'ticket_prices.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')

ringkas = pd.DataFrame({
    'baris': [len(x) for x in (train, th, te, movies, hol, prices, sample)],
    'tanggal awal': [x.date_show.min().date() if 'date_show' in x else (x.date.min().date() if 'date' in x else '') for x in (train, th, te, movies, hol, prices, sample)],
    'tanggal akhir': [x.date_show.max().date() if 'date_show' in x else (x.date.max().date() if 'date' in x else '') for x in (train, th, te, movies, hol, prices, sample)],
}, index=['train', 'test_history', 'test', 'movies', 'holidays', 'ticket_prices', 'sample_submission'])
display(ringkas)

kunci = ['date_show', 'cinema_ids', 'movie_title']
print('\nAudit kualitas')
for nm, d in (('train', train), ('test_history', th)):
    print('  %-13s duplikat kunci %d | nilai kosong %d | occupation di luar 0-100: %d | total_ticket = 0: %d | tiket > 0 dengan show = 0: %d' % (
        nm, d.duplicated(kunci).sum(), d.isna().sum().sum(), (~d.occupation_rate.between(0, 100)).sum(),
        (d.total_ticket == 0).sum(), ((d.total_ticket > 0) & (d.total_show == 0)).sum()))
print('  satu klaster = satu kota (train+history):', (pd.concat([train, th]).groupby('cinema_ids').city_name.nunique() == 1).all())
print('  kota di train/test yang tidak punya harga tiket:', set(pd.concat([train, th, te]).city_name) - set(prices.city_name) or 'tidak ada')
print('  judul di train/test_history tanpa metadata movies.csv:',
      len({judul_dasar(t).upper() for t in pd.concat([train, th]).movie_title.unique()} - set(movies.original_title.str.upper())))

Data bersih: tidak ada duplikat, nilai kosong, atau occupancy di luar rentang. Satu fakta penting: **`total_ticket` tidak pernah bernilai 0**.
Sumber hanya memuat baris transaksi, sedangkan aturan lomba menyebut hari tanpa transaksi bernilai 0. Itu menentukan cara target dibangun di bagian 3.

## 2. Membongkar aturan pembentukan data uji

Tiga pertanyaan menentukan seluruh pendekatan: apa itu D1, pasangan mana yang masuk `test.csv`, dan apakah target bisa bernilai nol.

In [ ]:
d1_test = th.groupby('movie_title').date_show.min()
span = th.groupby('movie_title').date_show.agg(lambda s: (s.max() - s.min()).days)
print('test_history: %d film | rentang hari per film: %s' % (len(d1_test), span.value_counts().to_dict()))
print('baris test dari film yang tidak ada di train: %.1f%%' % (100 * (~te.movie_title.isin(train.movie_title)).mean()))

te['h'] = (te.date_show - te.movie_title.map(d1_test)).dt.days
print('test.csv     : %d baris | %d pasangan film-klaster | baris per pasangan: %s | horizon (hari sejak D1): %s' % (
    len(te), te.groupby(['movie_title', 'cinema_ids']).ngroups,
    te.groupby(['movie_title', 'cinema_ids']).size().value_counts().to_dict(), sorted(te.h.unique())))

hh = th.assign(h=(th.date_show - th.movie_title.map(d1_test)).dt.days)
hist_pair = hh.pivot_table(index=['movie_title', 'cinema_ids'], columns='h', values='total_ticket', aggfunc='sum').reindex(columns=range(3)).fillna(0)
pair_te = te.groupby(['movie_title', 'cinema_ids']).size().index
masuk = hist_pair.index.isin(pair_te)
print('\nPasangan di test_history: %d | masuk test.csv: %d | dibuang: %d' % (len(hist_pair), masuk.sum(), (~masuk).sum()))
print('  yang masuk test.csv dan aktif di D3  : %d dari %d' % ((hist_pair[masuk][2] > 0).sum(), masuk.sum()))
print('  yang dibuang dan aktif di D3         : %d dari %d' % ((hist_pair[~masuk][2] > 0).sum(), (~masuk).sum()))

**Temuan.** `test_history` selalu memuat tepat tiga hari per film, `test.csv` adalah tujuh hari berikutnya, dan sebuah pasangan film-klaster masuk
`test.csv` **kalau dan hanya kalau ia masih punya transaksi di D3**. Konsekuensinya:

- Pasangan yang berhenti tayang di tengah D4-D10 tetap ada di test dengan target 0. Target nol itu nyata, bukan data hilang.
- Training set semu harus memakai aturan seleksi yang sama: pasangan hidup di D3.

In [ ]:
nc = train.groupby(['movie_title', 'date_show']).cinema_ids.nunique()
d1_min = train.groupby('movie_title').date_show.min()

def d1_rilis(film):
    # hari rilis lebar: hari pertama dengan jumlah klaster aktif >= 50% puncaknya dalam 21 hari pertama
    s = nc.loc[film]
    s = s[s.index <= s.index.min() + pd.Timedelta(days=20)]
    return s[s >= 0.5 * s.max()].index.min()

d1_rl = pd.Series({f: d1_rilis(f) for f in d1_min.index})
geser = (d1_rl - d1_min).dt.days
print('Film di train: %d | D1 bergeser dari tanggal transaksi pertama: %d (%.0f%%)' % (len(geser), (geser > 0).sum(), 100 * (geser > 0).mean()))
k_naif = np.median([nc.loc[f].loc[d1_min[f]] for f in d1_min.index])
k_rilis = np.median([nc.loc[f].loc[d1_rl[f]] for f in d1_min.index])
k_test = hh[hh.h == 0].groupby('movie_title').cinema_ids.nunique().median()
print('median klaster aktif di D1: train dengan tanggal transaksi pertama %.0f | train dengan hari rilis lebar %.0f | test %.0f' % (k_naif, k_rilis, k_test))

import matplotlib.dates as mdates
kandidat = geser[(geser >= 2) & (geser <= 7) & (d1_min > pd.Timestamp('2025-04-01'))].index
contoh = nc.groupby(level=0).max()[kandidat].sort_values(ascending=False).index[:2]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for a, f in zip(ax, contoh):
    s = nc.loc[f]; s = s[(s.index >= d1_min[f]) & (s.index <= d1_rl[f] + pd.Timedelta(days=6))]
    warna = [C_TEST if t == d1_min[f] else (C_ACC if t == d1_rl[f] else C_TRAIN) for t in s.index]
    a.bar(s.index, s.values, color=warna, width=.8); a.set_ylim(0, s.max() * 1.4)
    a.set_title(f[:44] + ': klaster yang menayangkan per hari', loc='left', fontsize=9.5)
    a.xaxis.set_major_formatter(mdates.DateFormatter('%d %b')); plt.setp(a.get_xticklabels(), rotation=30, ha='right')
ax[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=C_TEST), plt.Rectangle((0, 0), 1, 1, color=C_ACC)],
             labels=['tanggal transaksi pertama', 'hari rilis lebar (D1)'], fontsize=8, loc='upper left')
plt.tight_layout(); plt.show()

**D1 adalah hari rilis lebar, bukan tanggal transaksi pertama.** Banyak film di train diawali sneak preview atau premiere di satu atau dua klaster,
lalu rilis lebar beberapa hari kemudian. Di `test_history` hari seperti itu tidak ada: setiap film langsung dimulai pada hari rilis lebar.
Median klaster aktif di D1 hanya 1 bila D1 diambil dari tanggal transaksi pertama, 46 bila dari hari rilis lebar, dan 70 di test (angka di atas).
Memakai tanggal transaksi pertama menggeser horizon sehingga rasio D4-D10 terhadap skala tidak sebanding dengan test.
Sisa selisih 46 lawan 70 adalah rilis bertahap di sebagian film latih; itu ditangani lewat pembobotan komposisi lebar rilis (bagian 4 dan 5).

## 3. Training set semu dan sifat target

`train.csv` hanya riwayat transaksi, tanpa penanda D1-D10. Label dibangun dengan aturan yang sama dengan test:

| Aturan | Isi |
|---|---|
| D1 | hari rilis lebar (bagian 2) |
| Sensor kiri | film yang D1 asli-nya sebelum 1 April 2025 dibuang, karena tanggal rilisnya tidak diketahui |
| Sensor kanan | film yang D10-nya melewati 30 September 2025 dibuang |
| Pasangan | film-klaster yang aktif di D3 (aturan seleksi test) |
| Hari kosong | bernilai 0 |
| Skala | rata-rata tiket D1-D3 per pasangan |
| Target | tiket D4-D10 per pasangan; model memprediksi **rasio target / skala** |

In [ ]:
H = list(range(3, 10))                       # indeks hari D4..D10 sejak D1 (D1 = 0)
film_ok = d1_rl[(d1_min > '2025-04-01') & (d1_rl + pd.Timedelta(days=9) <= '2025-09-30')]

d = train[train.movie_title.isin(film_ok.index)].copy()
d['h'] = (d.date_show - d.movie_title.map(film_ok)).dt.days
d = d[(d.h >= 0) & (d.h < 10)]
U = (d.pivot_table(index=['movie_title', 'cinema_ids'], columns='h', values='total_ticket', aggfunc='sum')
       .reindex(columns=range(10), fill_value=0).fillna(0))
U = U[U[2] > 0]                              # hidup di D3, seperti seleksi test.csv
U['scale'] = U[[0, 1, 2]].mean(axis=1)
U['d1'] = U.index.get_level_values(0).map(film_ok)

T = (hh.pivot_table(index=['movie_title', 'cinema_ids'], columns='h', values='total_ticket', aggfunc='sum')
       .reindex(columns=range(3), fill_value=0).fillna(0))
T['scale'] = T[[0, 1, 2]].mean(axis=1)
T = T.loc[pair_te]

film = U.index.get_level_values(0).values
Y = U[H].values.astype(float)
S = U.scale.values.astype(float)
R = Y / S[:, None]
N = len(U)
print('film di train: %d | tersensor kiri: %d | tersensor kanan: %d | dipakai: %d' % (
    len(d1_min), (d1_min <= '2025-04-01').sum(), ((d1_min > '2025-04-01') & (d1_rl + pd.Timedelta(days=9) > '2025-09-30')).sum(), len(film_ok)))
print('training set semu: %d pasangan dari %d film | test: %d pasangan dari %d film' % (N, len(np.unique(film)), len(T), T.index.get_level_values(0).nunique()))

nh = lambda P: (P[[0, 1, 2]] > 0).sum(axis=1).value_counts(normalize=True).sort_index().round(3).to_dict()
print('\nBagian pasangan menurut jumlah hari aktif di D1-D3 (1, 2, 3 hari):')
print('  training set semu :', nh(U))
print('  test              :', nh(T))

# pembanding: bila D1 = tanggal transaksi pertama (definisi lama)
dn = train[train.movie_title.isin(d1_min[d1_min > '2025-04-01'].index)].copy()
dn['h'] = (dn.date_show - dn.movie_title.map(d1_min)).dt.days
Un = (dn[(dn.h >= 0) & (dn.h < 3)].pivot_table(index=['movie_title', 'cinema_ids'], columns='h', values='total_ticket', aggfunc='sum')
        .reindex(columns=range(3), fill_value=0).fillna(0))
Un = Un[Un[2] > 0]
print('  D1 = transaksi pertama:', nh(Un))

Dengan D1 hari rilis, komposisi hari aktif di D1-D3 pada training set semu hampir sama dengan test (sekitar 91% pasangan aktif di ketiga hari).
Dengan tanggal transaksi pertama, angkanya jauh lebih rendah (baris `D1 = transaksi pertama` di atas). Itu bukti langsung bahwa definisi D1 yang benar adalah hari rilis lebar.

In [ ]:
p0 = (Y == 0).mean(axis=0)
med = np.median(R, axis=0); mean = R.mean(axis=0)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
ax[0].bar([f'D{h+1}' for h in H], p0, color=C_TEST)
ax[0].axhline(.5, color='k', ls='--', lw=1)
ax[0].set_title('Peluang target = 0 naik tajam: %.2f di D4, %.2f di D10' % (p0[0], p0[-1]), loc='left')
ax[1].plot(range(7), med, 's-', color=C_TRAIN, lw=2, label='median')
ax[1].plot(range(7), mean, 'o-', color=C_ACC, lw=2, label='rata-rata')
ax[1].set_xticks(range(7)); ax[1].set_xticklabels([f'D{h+1}' for h in H]); ax[1].legend()
ax[1].set_title('Median di bawah rata-rata: ekor kanan panjang', loc='left')
ax[2].hist(np.log10(np.clip(R[R > 0], 1e-3, None)), bins=60, color=C_TRAIN)
ax[2].set_title('Rasio target / skala (sel > 0), skala log10', loc='left')
plt.tight_layout(); plt.show()
print('median rasio D4..D10 :', np.round(med, 2))
print('rata-rata rasio      :', np.round(mean, 2))

**Konsekuensi untuk model.** Metrik adalah `|y - prediksi| / skala`, yaitu galat absolut. Prediksi yang meminimalkan galat absolut adalah **median**,
dan di D9-D10 lebih dari separuh target bernilai nol, sehingga median di sana adalah nol. Rata-rata akan menarik prediksi ke atas oleh segelintir sel
yang sangat ramai. Bagian 8 mengukur selisihnya.

## 4. Pergeseran data latih ke data uji

Tiga perbedaan yang diukur, karena menentukan pembobotan validasi dan koreksi musim.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
ncl_tr = pd.Series(1, index=range(N)).groupby(film).transform('sum').values
ncl_te = pd.Series(1, index=range(len(T))).groupby(T.index.get_level_values(0).values).transform('sum').values

for v, c, nm in ((S, C_TRAIN, 'training set semu'), (T.scale.values, C_TEST, 'test')):
    x = np.sort(v); ax[0].plot(x, np.arange(1, len(x) + 1) / len(x), color=c, lw=2, label=nm)
ax[0].set_xscale('log'); ax[0].set_xlabel('skala pasangan (tiket per hari)'); ax[0].legend()
ax[0].set_title('Pasangan test jauh lebih kecil (median %.0f vs %.0f)' % (np.median(T.scale), np.median(S)), loc='left')

bins = [0, 40, 80, 999]; lab = ['< 40', '40-79', '>= 80']
share = lambda n: pd.Series(pd.cut(n, bins, right=False, labels=lab)).value_counts(normalize=True).sort_index().values
x = np.arange(3)
ax[1].bar(x - .2, share(ncl_tr), .4, color=C_TRAIN, label='training set semu'); ax[1].bar(x + .2, share(ncl_te), .4, color=C_TEST, label='test')
ax[1].set_xticks(x); ax[1].set_xticklabels(lab); ax[1].set_xlabel('jumlah klaster yang menayangkan film'); ax[1].legend()
ax[1].set_title('Test didominasi film rilis lebar', loc='left')

import matplotlib.dates as mdates
n2d = mdates.date2num
hd = hol[hol.holiday_tipe == 'holiday']
jend = {1: (pd.Timestamp('2025-04-01'), pd.Timestamp('2025-09-30'), C_TRAIN), 0: (pd.Timestamp('2025-10-01'), pd.Timestamp('2026-03-27'), C_TEST)}
for y, (a_, b_, c_) in jend.items():
    ax[2].barh([y], [(b_ - a_).days], left=n2d(a_), color=c_, height=.5, alpha=.3)
    for dt in hd.date[(hd.date >= a_) & (hd.date <= b_)]:
        ax[2].plot([n2d(dt)] * 2, [y - .3, y + .3], 'k-', lw=1.3)
def pita(y, a_, b_, warna, nama):
    ax[2].barh([y], [(pd.Timestamp(b_) - pd.Timestamp(a_)).days], left=n2d(pd.Timestamp(a_)), color=warna, height=.22, label=nama)
pita(1, '2025-06-28', '2025-07-12', C_ACC, 'libur sekolah')
pita(0, '2025-12-20', '2026-01-04', C_ACC, '_'); pita(0, '2026-03-21', '2026-03-27', C_ACC, '_')
pita(0, '2026-02-19', '2026-03-20', '#f59e0b', 'Ramadan 1447 H')
ax[2].set_yticks([0, 1]); ax[2].set_yticklabels(['test', 'train']); ax[2].grid(False)
ax[2].xaxis.set_major_locator(mdates.MonthLocator(interval=2)); ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax[2].legend(fontsize=8, loc='upper center', ncol=2, bbox_to_anchor=(.5, -.12))
n_tr = hd.date.between('2025-04-01', '2025-09-30').sum(); n_te = hd.date.between('2025-10-01', '2026-03-27').sum()
ax[2].set_title('Garis: libur nasional (%d di train, %d di test)' % (n_tr, n_te), loc='left')
plt.tight_layout(); plt.show()

print('pasangan dari film >= 60 klaster: training set semu %.0f%% | test %.0f%%' % (100 * (ncl_tr >= 60).mean(), 100 * (ncl_te >= 60).mean()))
print('\nLibur nasional di jendela test dan banyak baris test pada tanggal itu:')
for _, r in hd[hd.date >= '2025-10-01'].iterrows():
    print('   %s  %-40s %5d baris' % (r.date.date(), r.holiday_name, (te.date_show == r.date).sum()))

1. **Skala.** Pasangan test kira-kira separuh ukuran pasangan latih. Seperdelapan terkecil dari pasangan latih berisi sekitar 29% pasangan test (lihat grafik dan angka di atas).
   Pasangan kecil punya galat relatif terbesar, jadi validasi harus memberi bobot lebih ke sana.
2. **Lebar rilis.** Film di test lebih sering rilis lebar (80% pasangan dari film dengan sedikitnya 60 klaster). Pasangan kecil dari film rilis lebar
   adalah sel dengan biaya tertinggi.
3. **Kalender.** Test memuat Natal, Tahun Baru, Imlek, Nyepi, Idulfitri, dan Ramadan. Data latih memuat Idulfitri 2025 (hanya sebagai pasar umum, bukan sebagai
   riwayat film), libur sekolah Juni-Juli, dan Idul Adha. Bagian 9 memakai padanan itu untuk dua koreksi musim.

## 5. Protokol validasi

Kandidat dinilai dengan tiga ukuran, dan sebuah perubahan dianggap berhasil hanya bila membaik di ketiganya.

| Ukuran | Cara |
|---|---|
| **CV** | GroupKFold 5 lipatan **per film**. Satu film tidak pernah ada di latih dan uji sekaligus, karena 92% baris test adalah film yang tidak ada di train |
| **OOT** | Latih pada film dengan D1 sebelum 1 Juli, uji pada film D1 mulai 1 Juli (Apr-Jun ke Jul-Sep) |
| **Berbobot komposisi test** | Setiap pasangan diberi bobot rasio pangsa test terhadap pangsa latih pada sel (8 kelompok skala x 3 kelompok lebar rilis) |

Ukuran ketiga disebut **CVw-2D**. Ukuran ini dikalibrasi pada dua skor leaderboard: model dengan CVw-2D 0,506 mendapat 0,51, dan model dengan 0,483 mendapat 0,49.
Bobot untuk melatih model hanya memakai kelompok skala (8 kelompok), sedangkan CVw-2D dipakai hanya untuk menilai.

In [ ]:
film_u = np.unique(film)
rng = np.random.RandomState(0)
fold = dict(zip(film_u, rng.permutation(len(film_u)) % 5))
fk = np.array([fold[f] for f in film])
f_awal = (U.d1 < '2025-07-01').values; e_akhir = ~f_awal

# pembobot komposisi test
St = T.scale.values
batas = np.quantile(np.log10(S), np.linspace(0, 1, 9)[1:-1])
bin_tr, bin_te = np.digitize(np.log10(S), batas), np.digitize(np.log10(St), batas)
W_LATIH = (np.bincount(bin_te, minlength=8) / len(St)) / (np.bincount(bin_tr, minlength=8) / N)
W_LATIH = W_LATIH[bin_tr]                                          # bobot melatih: kelompok skala saja
sel_tr = bin_tr * 3 + np.digitize(ncl_tr, [40, 80]); sel_te = bin_te * 3 + np.digitize(ncl_te, [40, 80])
rasio = np.where(np.bincount(sel_tr, minlength=24) > 0,
                 (np.bincount(sel_te, minlength=24) / len(St)) / np.maximum(np.bincount(sel_tr, minlength=24) / N, 1e-9), 0)
W_NILAI = np.clip(rasio, .2, 6)[sel_tr]                             # bobot menilai: skala x lebar rilis

def skor(P, mask=None):
    # MASE resmi: |y - p| / max(skala, 1), dirata-rata atas sel; mask memilih pasangan
    c = np.abs(Y - P) / np.maximum(S, 1)[:, None]
    m = np.ones(N, bool) if mask is None else mask
    return c[m].mean(), np.average(c[m].mean(axis=1), weights=W_NILAI[m])

print('bobot melatih per kelompok skala :', np.round(np.unique(W_LATIH), 2))
print('ukuran sampel efektif bobot menilai: %.0f dari %d pasangan' % (W_NILAI.sum() ** 2 / (W_NILAI ** 2).sum(), N))

## 6. Rekayasa fitur

Satu fungsi membangun fitur untuk training set semu dan test dari baris D1-D3 saja, supaya kedua sisi tidak bisa berbeda definisi.
Fitur dikelompokkan menurut pertanyaan yang dijawab.

| Kelompok | Fitur | Alasan |
|---|---|---|
| Riwayat pasangan | `ls` (log10 skala), `r0, r1, r2` (D1-D3 dibagi skala), `nh`, `pat` | tingkat dan bentuk penjualan awal; `pat` menandai pasangan yang baru buka di D2 atau D3 |
| Komposisi hari D1-D3 | `d1dow`, `noff_hist`, `off0-off2` | film yang rilis Rabu tidak punya akhir pekan di D1-D3, film yang rilis Jumat punya dua; skala dua film itu tidak sebanding tanpa fitur ini |
| Hari target | horizon, hari dalam minggu, akhir pekan, libur nasional (`hol`) | pola mingguan dan libur |
| Harga relatif (`prel`) | harga tiket hari target dibagi rata-rata harga D1-D3 di kota yang sama | `holidays.day_tipe` dan `ticket_prices.price_day` memakai tiga kategori yang sama (Weekday, Friday, Weekend); hari libur diberi harga akhir pekan |
| Kalender kumulatif | `cumoff`, `sinceoff` | jumlah hari libur sampai hari target dan jarak dari hari libur terakhir |
| Film lokal | `lokal` | film Indonesia turun lebih tajam setelah akhir pekan pertama; sumbernya dijelaskan di bawah |
| Tabel median | `tab` | prediksi tabel (kelompok skala x horizon x hari) sebagai fitur tambahan |

Cuti bersama (SKB 3 Menteri) tidak ada di `holidays.csv`, jadi ditambahkan sebagai libur hari kerja.

In [ ]:
D_ = lambda s: np.datetime64(s, 'D')
HOL = hol.loc[hol.holiday_tipe == 'holiday', 'date'].values.astype('datetime64[D]')
# Cuti bersama: SKB 3 Menteri 2025 (terbit Okt 2024, revisi 18 Agt 2025) dan SKB 3 Menteri 2026 (ditandatangani 19 Sep 2025)
CUTI = np.array([D_(x) for x in ['2025-04-02', '2025-04-03', '2025-04-04', '2025-04-07', '2025-05-13', '2025-05-30', '2025-06-09',
                                 '2025-08-18', '2025-12-26', '2026-02-16', '2026-03-18', '2026-03-20', '2026-03-23', '2026-03-24']])
TP = prices.pivot_table(index='city_name', columns='price_day', values='ceil')
KOTA = pd.concat([train[['cinema_ids', 'city_name']], th[['cinema_ids', 'city_name']]]).drop_duplicates('cinema_ids').set_index('cinema_ids').city_name

def kalender(t):
    flat = t.ravel()
    dw = pd.DatetimeIndex(flat).dayofweek.values
    off = (dw >= 5) | np.isin(flat, HOL) | np.isin(flat, CUTI)
    ptype = np.where(off, 2, np.where(dw == 4, 1, 0))             # 0 Weekday, 1 Friday, 2 Weekend (libur = harga akhir pekan)
    return {k: v.reshape(t.shape) for k, v in dict(dow=dw, hol=np.isin(flat, HOL).astype(int), off=off.astype(int), ptype=ptype).items()}

def harga(kota, ptype):
    return np.take_along_axis(TP.reindex(kota)[['Weekday', 'Friday', 'Weekend']].values, ptype, axis=1)

def fitur(hist, pairs, d1map, lokal):
    g = hist.groupby(['movie_title', 'cinema_ids', 'h']).total_ticket.sum().unstack().reindex(pairs).reindex(columns=range(3)).fillna(0)
    y0, y1, y2 = (g[i].values.astype(float) for i in range(3))
    sk = (y0 + y1 + y2) / 3
    fl = pairs.get_level_values(0).values
    kota = KOTA.reindex(pairs.get_level_values(1)).values
    d1 = np.asarray(pd.Series(fl).map(d1map).values, dtype='datetime64[D]')
    kh = kalender(d1[:, None] + np.arange(3).astype('timedelta64[D]')[None, :])
    kt = kalender(d1[:, None] + np.array(H, dtype='timedelta64[D]')[None, :])
    ph, pt = harga(kota, kh['ptype']), harga(kota, kt['ptype'])
    sejak = np.zeros_like(kt['off']); c = np.where(kh['off'][:, 2] == 1, 0, 9)
    for j in range(7):
        c = np.where(kt['off'][:, j] == 1, 0, c + 1); sejak[:, j] = c
    F = pd.DataFrame({'ls': np.log10(sk), 'r0': y0 / sk, 'r1': y1 / sk, 'r2': y2 / sk, 'nh': (g > 0).sum(axis=1).values,
                      'pat': np.where((y0 > 0) & (y1 > 0), 0, np.where(y0 > 0, 1, np.where(y1 > 0, 2, 3))),
                      'lokal': pd.Series([judul_dasar(f).upper() for f in fl]).map(lokal).values,
                      'd1dow': kh['dow'][:, 0], 'noff_hist': kh['off'].sum(axis=1),
                      'off0': kh['off'][:, 0], 'off1': kh['off'][:, 1], 'off2': kh['off'][:, 2]})
    M = dict(dow=kt['dow'], hol=kt['hol'], off=kt['off'], prel=pt / ph.mean(axis=1, keepdims=True),
             cumoff=np.cumsum(kt['off'], axis=1), sinceoff=np.minimum(sejak, 9))
    return F, M, sk, y2

### Fitur film lokal

Sumber label: halaman Wikipedia *Daftar film Indonesia tahun 2025*, **revisi 27901699 per 30 September 2025** (diunduh dari revisi tetap itu, bukan halaman terbaru).
Sebuah film berlabel lokal bila tercantum di daftar itu, atau berbagi sedikitnya satu sutradara, penulis, produser, atau pemeran (dari `movies.csv`) dengan
film latih yang tercantum. Jalur kedua menangkap film yang rilis setelah 30 September 2025 dan belum ada di daftar. Film latih tidak menjadi bukti bagi dirinya sendiri.

In [ ]:
WIKI_OLDID = 27901699
WIKI_FILE = EXT / 'wiki_daftar_film_indonesia_2025_oldid27901699.wikitext'
if not WIKI_FILE.exists():
    import requests
    r = requests.get('https://id.wikipedia.org/w/api.php', timeout=30, headers={'User-Agent': 'JOINTS2026-notebook/1.0'},
                     params=dict(action='query', prop='revisions', revids=WIKI_OLDID, rvprop='ids|timestamp|content', rvslots='main', format='json')).json()
    rev = list(r['query']['pages'].values())[0]['revisions'][0]
    assert rev['timestamp'] <= '2025-09-30T23:59:59Z', 'revisi melewati batas aturan lomba'
    WIKI_FILE.write_text(rev['slots']['main']['*'], encoding='utf-8')
wikitext = WIKI_FILE.read_text(encoding='utf-8')

def norm(s):
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r'\(film[^)]*\)', '', s, flags=re.I)
    return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()

judul_wiki = {norm(m.group(2) or m.group(1)) for m in re.finditer(r"''\[\[([^\]|]+)(?:\|([^\]]+))?\]\]''", wikitext)}
judul_wiki |= {norm(m.group(1)) for m in re.finditer(r"''([^'\[\]{}|\n]{2,80})''", wikitext)}
judul_wiki.discard('')

movies['k'] = movies.original_title.str.upper().str.strip()
orang = {r.k: {x.strip().lower() for c in ['director', 'producer', 'writer', 'casts'] for x in str(r[c]).split(',') if x.strip() and x.strip().lower() != 'nan'}
         for _, r in movies.iterrows()}
di_wiki = {k: norm(k) in judul_wiki for k in orang}
benih = {judul_dasar(f).upper() for f in train.movie_title.unique()} & {k for k, v in di_wiki.items() if v}
bukti = Counter(p for k in benih for p in orang[k])
LOKAL = pd.Series({k: float(di_wiki[k] or sum((bukti[p] - (k in benih)) > 0 for p in s) >= 1) for k, s in orang.items()})
print('judul di revisi Wikipedia: %d | film latih berlabel (benih): %d | film berlabel lokal di movies.csv: %d dari %d' % (len(judul_wiki), len(benih), LOKAL.sum(), len(LOKAL)))

In [ ]:
F_tr, M_tr, S_chk, y2_tr = fitur(d[d.h < 3], U.index, film_ok, LOKAL)
F_te, M_te, S_te, y2_te = fitur(hh[hh.h < 3], pair_te, d1_test, LOKAL)
assert np.allclose(S_chk, S) and np.allclose(S_te, St), 'fitur tidak cocok dengan training set semu atau test'
print('pangsa pasangan film lokal: training set semu %.1f%% | test %.1f%%' % (100 * np.nanmean(F_tr.lokal), 100 * np.nanmean(F_te.lokal)))
print('fitur tingkat pasangan:', list(F_tr.columns))
print('fitur per hari target :', list(M_tr))

## 7. Model

**Target.** Rasio `tiket / skala` untuk tiap pasangan dan horizon, dengan prediksi akhir `rasio x skala` dibulatkan ke bilangan bulat.
**Loss.** L1 (median), sesuai metrik. **Bobot latih.** Komposisi skala test (bagian 5).

Prediksi akhir adalah ensemble dari empat komponen:

| Komponen | Isi |
|---|---|
| M2 | HGB (loss L1) terpisah untuk film lokal dan film impor, dengan regularisasi kuat (31 daun, minimum 150 sampel, L2 = 10) |
| M3 | XGBoost dengan `reg:absoluteerror` |
| Tabel | median rasio per kelompok skala (5) x horizon x hari target, diencerkan ke median kelompok bila sel kecil, ditambah aturan pasangan yang baru buka di D2 atau D3 (tiket D3 dipakai sebagai level) |
| Gerbang nol | classifier peluang target = 0; prediksi dinolkan bila peluangnya di atas 0,55 |

`prediksi = nol bila gerbang > 0,55, selain itu bulat(0,8 x rata-rata(M2, M3) + 0,2 x Tabel)`.
Hiperparameter dipilih dari kisi kecil yang dikunci sebelum dijalankan dan diterima hanya bila membaik di ketiga ukuran validasi (bagian 11).

In [ ]:
KOL = ['ls', 'r0', 'r1', 'r2', 'nh', 'pat', 'lokal', 'd1dow', 'noff_hist', 'off0', 'off1', 'off2']
MKOL = ['dow', 'hol', 'prel', 'cumoff', 'sinceoff']
HP = dict(loss='absolute_error', max_iter=250, learning_rate=0.05, max_leaf_nodes=20, min_samples_leaf=60, l2_regularization=1.0, random_state=0)
HP_LOKAL = dict(HP, max_leaf_nodes=31, min_samples_leaf=150, l2_regularization=10.0)
N0 = 30                                    # pengenceran sel kecil ke median kelompok: n / (n + N0)
rnd = lambda x: np.maximum(np.floor(x + .5), 0)

class Blok:
    # satu sisi data (training set semu atau test): fitur pasangan, fitur per hari, skala, tiket D3, pola hari aktif
    def __init__(self, F, M, sk, y2):
        self.F, self.M, self.S, self.y2 = F, M, sk, y2
        self.pat = F['pat'].values
        self.lokal = np.nan_to_num(F['lokal'].values) == 1

TR, TE = Blok(F_tr, M_tr, S, y2_tr), Blok(F_te, M_te, St, y2_te)

def fit_tab(sk, rasio, dow, nb):
    ls = np.log10(sk); bt = np.quantile(ls, np.linspace(0, 1, nb + 1)[1:-1]); k = np.digitize(ls, bt)
    base = np.zeros((nb, 7)); tab = np.zeros((nb, 7, 7))
    for q in range(nb):
        for j in range(7):
            m = k == q
            base[q, j] = np.median(rasio[m, j]) if m.any() else np.median(rasio[:, j])
            for w in range(7):
                mw = m & (dow[:, j] == w); n = int(mw.sum()); lam = n / (n + N0)
                tab[q, j, w] = lam * (np.median(rasio[mw, j]) if n else 0) + (1 - lam) * base[q, j]
    return bt, tab

def tab_ratio(sk, dow, bt, tab):
    k = np.digitize(np.log10(sk), bt)
    return np.stack([tab[k, j, dow[:, j]] for j in range(7)], axis=1)

def matriks(blok, idx, tab):
    X = []
    Fs = blok.F[KOL].values[idx]
    for j in range(7):
        bagian = [Fs, np.full((len(idx), 1), j), (blok.M['dow'][idx, j] >= 5)[:, None]]
        bagian += [blok.M[c][idx, j][:, None] for c in MKOL] + [tab[:, j][:, None]]
        X.append(np.hstack(bagian).astype(float))
    return np.vstack(X)

def siapkan(a, blok, idx):
    # tabel median dilatih pada baris latih a, lalu dipakai sebagai fitur untuk baris latih dan baris sasaran
    bt, tab = fit_tab(S[a], R[a], TR.M['dow'][a], 3)
    Xa = matriks(TR, np.where(a)[0], tab_ratio(S[a], TR.M['dow'][a], bt, tab))
    Xt = matriks(blok, idx, tab_ratio(blok.S[idx], blok.M['dow'][idx], bt, tab))
    return Xa, Xt

def regresi(algo, a, blok, idx, loss=None):
    Xa, Xt = siapkan(a, blok, idx)
    ya = np.concatenate([R[a, j] for j in range(7)]); sw = np.tile(W_LATIH[a], 7)
    if algo == 'xgb':
        m = xgb.XGBRegressor(objective='reg:absoluteerror', n_estimators=600, learning_rate=0.03, max_depth=6, min_child_weight=100, subsample=0.8,
                             colsample_bytree=0.8, reg_lambda=5.0, tree_method='hist', n_jobs=8, random_state=0)
    else:
        m = HGB(**({**HP_LOKAL} if algo == 'hgb_lokal' else {**HP, **({'loss': loss} if loss else {})}))
    return m.fit(Xa, ya, sample_weight=sw).predict(Xt).reshape(7, -1).T

def gerbang(a, blok, idx):
    Xa, Xt = siapkan(a, blok, idx)
    c = HGC(max_iter=200, learning_rate=0.05, max_leaf_nodes=12, min_samples_leaf=60, l2_regularization=1.0, random_state=0)
    c.fit(Xa, (np.concatenate([Y[a, j] for j in range(7)]) == 0).astype(int), sample_weight=np.tile(W_LATIH[a], 7))
    return c.predict_proba(Xt)[:, 1].reshape(7, -1).T

def tabel(a, blok, idx, nb=5):
    bt, tab = fit_tab(S[a], R[a], TR.M['dow'][a], nb)
    P = rnd(tab_ratio(blok.S[idx], blok.M['dow'][idx], bt, tab) * blok.S[idx][:, None])
    for g in (2, 3):                                              # pasangan yang baru buka di D2 atau D3: level = tiket D3
        ma, mv = a & (TR.pat == g), blok.pat[idx] == g
        if ma.any() and mv.any():
            tg = np.array([np.median(Y[ma, j] / np.maximum(TR.y2[ma], 1e-9)) for j in range(7)])
            P[mv] = rnd(tg[None, :] * blok.y2[idx][mv][:, None])
    return P

def komponen(a, blok, idx):
    St_ = blok.S[idx]; lok_t = blok.lokal[idx]
    m2 = np.zeros((len(idx), 7))
    for seg in (True, False):                                     # M2: model terpisah film lokal dan impor
        a_s, t_s = a & (TR.lokal == seg), lok_t == seg
        if t_s.any():
            m2[t_s] = rnd(regresi('hgb_lokal', a_s, blok, idx[t_s]) * St_[t_s][:, None])
    m3 = rnd(regresi('xgb', a, blok, idx) * St_[:, None])
    return dict(m2=m2, m3=m3, tab=tabel(a, blok, idx), p0=gerbang(a, blok, idx))

def gabung(c, k=1.0, tau=0.55):
    mentah = 0.8 * ((c['m2'] + c['m3']) / 2) + 0.2 * c['tab']
    return np.where(c['p0'] > tau, 0, rnd(mentah * k))

## 8. Hasil validasi

Semua model dilatih ulang di tiap lipatan film dan pada pembagian waktu. Baris pertama sampai ketiga adalah pembanding sederhana,
baris berikutnya menambah satu komponen tiap kali.

In [ ]:
import time
t0 = time.time()
semua = np.arange(N)
def jalankan(fn):
    # fn(a, v) -> prediksi tiket (jumlah baris v, 7); mengembalikan prediksi CV (seluruh baris) dan prediksi OOT (baris e_akhir)
    cv = np.zeros_like(Y)
    for k in range(5):
        a, v = fk != k, fk == k
        cv[v] = fn(a, np.where(v)[0])
    oot = np.zeros_like(Y); oot[e_akhir] = fn(f_awal, np.where(e_akhir)[0])
    return cv, oot

KOMP = {}
def komp_fn(a, idx):
    key = (a.tobytes(), idx.tobytes())
    if key not in KOMP:
        KOMP[key] = komponen(a, TR, idx)
    return KOMP[key]

hasil = {}
hasil['Datar: prediksi = skala'] = (np.tile(rnd(S)[:, None], (1, 7)),) * 2
hasil['Median rasio per horizon'] = jalankan(lambda a, idx: rnd(np.median(R[a], axis=0)[None, :] * S[idx][:, None]))
def tab3(a, idx):
    bt, tab = fit_tab(S[a], R[a], TR.M['dow'][a], 3)
    return rnd(tab_ratio(S[idx], TR.M['dow'][idx], bt, tab) * S[idx][:, None])
hasil['Tabel kelompok skala x horizon x hari'] = jalankan(tab3)
hasil['Tabel 5 kelompok + pola pasangan baru buka'] = jalankan(lambda a, idx: tabel(a, TR, idx))
hasil['HGB tunggal, loss kuadrat (rata-rata)'] = jalankan(lambda a, idx: rnd(regresi('hgb', a, TR, idx, loss='squared_error') * S[idx][:, None]))
hasil['HGB tunggal, loss L1 (median)'] = jalankan(lambda a, idx: rnd(regresi('hgb', a, TR, idx) * S[idx][:, None]))
print('pembanding selesai (%.0f detik)' % (time.time() - t0))

In [ ]:
for k in range(5):
    komp_fn(fk != k, np.where(fk == k)[0])
komp_fn(f_awal, np.where(e_akhir)[0])
print('komponen ensemble selesai (%.0f detik)' % (time.time() - t0))
def pecah(i):
    # i = 0: CV, 1: OOT -> kumpulkan komponen menjadi matriks (N, 7)
    keluar = {k: np.zeros((N, 7)) for k in ('m2', 'm3', 'tab', 'p0')}
    if i == 0:
        for k in range(5):
            v = np.where(fk == k)[0]; c = KOMP[((fk != k).tobytes(), v.tobytes())]
            for n in keluar: keluar[n][v] = c[n]
    else:
        v = np.where(e_akhir)[0]; c = KOMP[(f_awal.tobytes(), v.tobytes())]
        for n in keluar: keluar[n][v] = c[n]
    return keluar
KC = [pecah(0), pecah(1)]
gab = lambda **kw: tuple(gabung(KC[i], **kw) for i in (0, 1))
hasil['M2: HGB terpisah lokal / impor'] = tuple(KC[i]['m2'] for i in (0, 1))
hasil['M3: XGBoost, loss absolut'] = tuple(KC[i]['m3'] for i in (0, 1))
hasil['Ensemble M2 + M3 + tabel, tanpa gerbang'] = gab(tau=9)
hasil['Ensemble lengkap dengan gerbang nol'] = gab()

tabel_hasil = pd.DataFrame({nm: (*skor(cv), *skor(oot, e_akhir), 100 * (cv == 0).mean()) for nm, (cv, oot) in hasil.items()},
                           index=['CV', 'CVw-2D', 'OOT', 'OOTw-2D', 'sel diprediksi 0 (%)']).T
display(tabel_hasil.round(4))
print('sel aktual bernilai 0 di training set semu: %.1f%%' % (100 * (Y == 0).mean()))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
y = np.arange(len(tabel_hasil))[::-1]
ax.barh(y, tabel_hasil['CVw-2D'], color=[C_ACC if n.startswith('Ensemble lengkap') else C_TRAIN for n in tabel_hasil.index])
ax.set_yticks(y); ax.set_yticklabels(tabel_hasil.index, fontsize=8.5)
ax.set_xlabel('CVw-2D (berbobot komposisi test, lebih kecil lebih baik)'); ax.set_xlim(.4, .95)
for yi, v in zip(y, tabel_hasil['CVw-2D']): ax.text(v + .005, yi, '%.3f' % v, va='center', fontsize=8.5)
ax.set_title('Galat turun dari pembanding sederhana ke ensemble; loss rata-rata jauh lebih buruk dari median', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

**Cara membaca.**

- **Median menang telak atas rata-rata.** Model yang sama dengan loss kuadrat jauh lebih buruk, karena metrik adalah galat absolut dan data punya ekor kanan panjang.
  Model L1 memprediksi nol untuk lebih banyak sel daripada model rata-rata (kolom terakhir tabel), mendekati pangsa nol yang sebenarnya.
- Tabel sederhana sudah kuat. Model pohon menurunkan galat dengan memakai komposisi hari D1-D3, harga relatif, dan label lokal.
- **Ensemble** M2 dan M3 lebih baik daripada keduanya sendiri, dan gerbang nol menurunkan galat sedikit lagi. Selisih dari satu langkah ke langkah berikutnya kecil
  (ratusan perseribu) tetapi konsisten di lipatan film.
- **CVw-2D untuk model akhir sebelum koreksi musim adalah sekitar 0,485**, dan model itu mendapat 0,49 di leaderboard (bagian 9).

## 9. Koreksi musim

Data latih (April-September) dan data uji (Oktober-Maret) berbeda musim. Model belajar dari hari-hari yang ada di train, jadi ia tidak bisa tahu
bahwa pekan Lebaran atau libur sekolah Desember lebih ramai daripada hari biasa kecuali ada padanannya di train.

**Prinsip.** Koreksi hanya diberikan bila (1) efeknya bisa diukur dari padanan di data latih, dan (2) aturan koreksinya bisa ditulis sebelum melihat skor.
Koreksi absolut per musim (misalnya "semua hari Ramadan dikali 1,2") tidak dipakai: bila riwayat D1-D3 dan hari target sama-sama di Ramadan,
rasionya tidak berubah. Koreksi hanya berarti bila **riwayat berada di luar musim dan hari target di dalamnya**.

### 9.1 Bukti dari data latih
Indeks pasar harian = tiket per klaster per hari dibagi rata-rata hari-minggu yang sama pada 21 April-31 Mei 2025 (masa tenang).

In [ ]:
pasar = train.groupby('date_show').agg(tk=('total_ticket', 'sum'), n=('cinema_ids', 'nunique'))
pasar['tpk'] = pasar.tk / pasar.n; pasar['dow'] = pasar.index.dayofweek
dasar = pasar.loc['2025-04-21':'2025-05-31'].groupby('dow').tpk.mean()
pasar['indeks'] = pasar.tpk / dasar.reindex(pasar.dow).values

hari_kerja_biasa = lambda x: x[(x.dow < 5) & ~x.index.isin(hol.loc[hol.holiday_tipe == 'holiday', 'date']) & ~x.index.isin(CUTI)]
jendela = {'2 pekan sebelum libur sekolah': ('2025-06-16', '2025-06-27'), 'libur sekolah (30 Jun-11 Jul)': ('2025-06-30', '2025-07-11'),
           '2 pekan sesudahnya': ('2025-07-14', '2025-07-25')}
sek_idx = {k: hari_kerja_biasa(pasar.loc[a:b]).indeks.mean() for k, (a, b) in jendela.items()}

fig, ax = plt.subplots(1, 2, figsize=(14, 3.9))
a = pasar.loc['2025-04-01':'2025-04-30']
ax[0].plot(a.index, a.indeks, 'o-', color=C_TRAIN, lw=1.8)
ax[0].axhline(1, color='k', lw=1, ls='--'); ax[0].axvline(pd.Timestamp('2025-03-31'), color=C_TEST, ls='--', label='1 Syawal 1446 H (31 Maret)')
ax[0].set_title('Lebaran 2025: pasar sampai 3,5 kali hari biasa', loc='left'); ax[0].legend(); ax[0].set_ylabel('indeks pasar')
import matplotlib.dates as mdates
ax[0].xaxis.set_major_formatter(mdates.DateFormatter('%d %b')); plt.setp(ax[0].get_xticklabels(), rotation=30, ha='right')
ax[1].bar(range(3), list(sek_idx.values()), color=[C_TRAIN, C_ACC, C_TRAIN])
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(['2 pekan\nsebelum', 'libur\nsekolah', '2 pekan\nsesudah']); ax[1].axhline(1, color='k', lw=1, ls='--')
ax[1].set_title('Libur sekolah Jun-Jul 2025: hari kerja lebih ramai', loc='left')
plt.tight_layout(); plt.show()

leb = pasar.loc['2025-04-01':'2025-04-06'].indeks
print('Lebaran 2025, hari ke-1 sampai ke-6 sesudah 1 Syawal: indeks', np.round(leb.values, 2), '| rata-rata %.2f' % leb.mean())
print('Hari kerja biasa di sekitar libur sekolah 2025:', {k: round(v, 2) for k, v in sek_idx.items()})
v_ = list(sek_idx.values())
print('Libur sekolah dibanding rata-rata dua pekan di sekelilingnya: x%.2f' % (v_[1] / np.mean([v_[0], v_[2]])))

### 9.2 Seberapa besar model meleset di sel libur sekolah?
Pengali terbaik dicari pada prediksi out-of-fold model akhir (bagian 8), yaitu pengali tunggal `k` yang meminimalkan galat berbobot pada sekelompok sel.
Nilai 1,00 berarti model sudah tepat di kelompok itu.

In [ ]:
mentah_cv = 0.8 * ((KC[0]['m2'] + KC[0]['m3']) / 2) + 0.2 * KC[0]['tab']
d1_arr = np.asarray(U.d1.values, dtype='datetime64[D]')
TGL_TR = d1_arr[:, None] + np.array(H, dtype='timedelta64[D]')[None, :]
sekolah_tr = np.isin(TGL_TR, np.arange(D_('2025-06-28'), D_('2025-07-13')))
w7 = np.repeat(W_NILAI[:, None], 7, 1)
def k_terbaik(mask, ks=np.round(np.arange(.8, 2.01, .05), 2)):
    c = [np.average((np.abs(Y - np.where(KC[0]['p0'] > .55, 0, rnd(mentah_cv * k))) / np.maximum(S, 1)[:, None])[mask], weights=w7[mask]) for k in ks]
    return ks[int(np.argmin(c))]

kelompok = {'hari kerja biasa, di luar libur sekolah': (M_tr['off'] == 0) & ~sekolah_tr,
            'hari kerja di libur sekolah (28 Jun-12 Jul)': (M_tr['off'] == 0) & sekolah_tr,
            'akhir pekan atau libur nasional': M_tr['off'] == 1}
print('%-48s %8s %10s' % ('kelompok sel', 'jumlah', 'k terbaik'))
for nm, m in kelompok.items():
    print('%-48s %8d %10.2f' % (nm, m.sum(), k_terbaik(m)))

Model memprediksi **terlalu rendah** pada hari kerja di libur sekolah (pengali terbaik jauh di atas 1,0), dan hampir tepat di akhir pekan dan hari libur nasional,
yang memang sudah dikenali lewat fitur `off`. Indeks pasar 9.1 menunjukkan kenaikan sekitar 1,3 kali, dan pengali terbaik di prediksi model sekitar 1,4. Koreksi yang dipakai (1,30) memilih nilai yang lebih rendah dari keduanya.

### 9.3 Koreksi yang dipakai pada test

| Koreksi | Faktor | Sel yang kena | Dasar |
|---|---|---|---|
| **Level umum** | 1,15 | semua | Kalibrasi dari dua skor leaderboard: prediksi x1,0 mendapat 0,49 dan prediksi x1,5 mendapat 0,519. Kurva biaya validasi digeser supaya cocok dengan kedua titik itu. Ini koreksi **paling lemah buktinya** karena bersandar pada leaderboard publik (10% data test) |
| **Libur sekolah** | 1,30 | hari kerja (bukan libur nasional) pada 20 Des 2025-3 Jan 2026 dan 21-26 Mar 2026 | Padanan Jun-Jul 2025 (9.1, 9.2). Tanggal dari SK Kalender Pendidikan DKI Jakarta 2025/2026 (PDF dibuat 23 Juni 2025) |
| **Lebaran** | 1,50 | semua hari target pada atau sesudah 21 Maret 2026 (1 Syawal 1447 H menurut `holidays.csv`) | Padanan Lebaran 2025 (9.1): indeks pasar 1,75-2,45 pada hari ke-1 sampai ke-6 (rata-rata 2,1). Pengali 1,5 sengaja di bawah indeks terukur, karena hari D1-D3 film-film itu (18-20 Maret) sudah libur dan ramai |

Koreksi Lebaran hanya mengenai film yang riwayat D1-D3-nya selesai sebelum Lebaran: tujuh film dengan D1 18 Maret 2026.

In [ ]:
K_LEVEL, K_SEKOLAH, K_LEBARAN = 1.15, 1.30, 1.50
LIBUR_SEKOLAH = np.concatenate([np.arange(D_('2025-12-20'), D_('2026-01-04')), np.arange(D_('2026-03-21'), D_('2026-03-27'))])
LEBARAN = D_('2026-03-21')

d1_te = np.asarray(pd.Series(pair_te.get_level_values(0)).map(d1_test).values, dtype='datetime64[D]')
TGL_TE = d1_te[:, None] + np.array(H, dtype='timedelta64[D]')[None, :]
faktor_sek = np.where(np.isin(TGL_TE, LIBUR_SEKOLAH) & (M_te['off'] == 0), K_SEKOLAH, 1.0)
faktor_leb = np.where(TGL_TE >= LEBARAN, K_LEBARAN, 1.0)

key = pd.MultiIndex.from_frame(te[['movie_title', 'cinema_ids']])
ip = pair_te.get_indexer(key); hj = te.h.values - 3
assert (ip >= 0).all() and ((hj >= 0) & (hj <= 6)).all()
print('baris test terkena koreksi libur sekolah : %d (%.1f%%)' % ((faktor_sek[ip, hj] > 1).sum(), 100 * (faktor_sek[ip, hj] > 1).mean()))
film_leb = te.movie_title[faktor_leb[ip, hj] > 1]
print('baris test terkena koreksi Lebaran       : %d (%.1f%%) dari %d film: %s' % (len(film_leb), 100 * len(film_leb) / len(te), film_leb.nunique(), ', '.join(sorted(film_leb.unique()))))

## 10. Submission

Model dilatih pada seluruh training set semu dan memprediksi semua pasangan test. Hasil per pasangan dipetakan ke baris `test.csv` lewat horizon.

In [ ]:
komp_te = komponen(np.ones(N, bool), TE, np.arange(len(St)))
mentah_te = 0.8 * ((komp_te['m2'] + komp_te['m3']) / 2) + 0.2 * komp_te['tab']
P_te = np.where(komp_te['p0'] > 0.55, 0, rnd(mentah_te * K_LEVEL * faktor_sek * faktor_leb))

sub = pd.DataFrame({'id': te.id.values, 'total_ticket': P_te[ip, hj].astype(int)})
assert len(sub) == len(sample) == 72611
assert (sub.id.values == sample.id.values).all(), 'urutan id berbeda dari sample_submission'
assert sub.total_ticket.notna().all() and (sub.total_ticket >= 0).all()
OUT = DATA / 'submission_juri_final.csv'
sub.to_csv(OUT, index=False)
print('ditulis %s | %d baris | %.1f%% bernilai 0 | rata-rata %.1f tiket' % (OUT, len(sub), 100 * (sub.total_ticket == 0).mean(), sub.total_ticket.mean()))

pembanding = DATA / 'submission_v7_patuh_lebaran_x150.csv'
if pembanding.exists():
    v7 = pd.read_csv(pembanding)
    print('identik dengan submission v7 yang mendapat 0,475 di leaderboard:', bool((v7.total_ticket.values == sub.total_ticket.values).all() and (v7.id.values == sub.id.values).all()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
rata = sub.assign(h=te.h.values).groupby('h').total_ticket.mean()
ax[0].plot([f'D{h+1}' for h in rata.index], rata.values, 'o-', color=C_TEST, lw=2)
ax[0].set_title('Prediksi test: rata-rata tiket per baris menurun dengan horizon', loc='left')
nol = sub.assign(h=te.h.values).groupby('h').total_ticket.apply(lambda s: (s == 0).mean())
ax[1].plot([f'D{h+1}' for h in nol.index], nol.values, 'o-', color=C_TEST, lw=2, label='prediksi test')
ax[1].plot([f'D{h+1}' for h in H], (Y == 0).mean(axis=0), 's--', color=C_TRAIN, lw=2, label='aktual training set semu')
nol_w = [np.average(Y[:, j] == 0, weights=W_NILAI) for j in range(7)]
ax[1].plot([f'D{h+1}' for h in H], nol_w, '^:', color=C_ACC, lw=2, label='aktual, berbobot komposisi test')
ax[1].legend(); ax[1].set_title('Pangsa nol per horizon', loc='left')
print('pangsa nol per horizon, berbobot komposisi test:', np.round(nol_w, 3), '| prediksi test:', np.round(nol.values, 3))
plt.tight_layout(); plt.show()

### Riwayat skor leaderboard

Skor adalah MASE di leaderboard publik, yang hanya mencakup sekitar 10% baris test.

| Submission | Perubahan | Skor |
|---|---|---|
| `submission_skala_kuartil` | median rasio per horizon dan kelompok skala; D1 = tanggal transaksi pertama | 0,573 |
| `submission_d1_rilis` | D1 = hari rilis lebar | 0,550 |
| `submission_internal_v2_gerbang` | HGB loss L1 + tabel + gerbang nol, bobot komposisi test. Fitur lokal dari TMDB* | 0,510 |
| `submission_final_v2` | + komposisi hari D1-D3, harga relatif, cuti bersama, ensemble HGB dan XGBoost. Fitur lokal dari TMDB* | 0,490 |
| `probe2_level_x1.5` | uji arah level (prediksi x1,5), bukan kandidat | 0,519 |
| `submission_v4_patuh_x115` | model yang sama dengan fitur lokal Wikipedia (patuh), level x1,15 | 0,486 |
| `submission_v5_patuh_libursekolah` | + koreksi libur sekolah | 0,483 |
| **`submission_v7_patuh_lebaran_x150`** | + koreksi Lebaran | **0,475** |

\* Dua submission awal memakai label bahasa asli film dari TMDB yang diambil setelah 30 September 2025. Itu melanggar aturan data eksternal, jadi diganti dengan
label dari Wikipedia revisi 30 September 2025 (bagian 6). Submission akhir tidak memakai TMDB.

## 11. Audit sumber eksternal, eksperimen yang gugur, dan batas

### 11.1 Audit sumber eksternal
Aturan lomba: sumber eksternal hanya boleh dipakai bila informasi atau versinya tersedia untuk publik paling lambat 30 September 2025 dan tanggalnya dapat dibuktikan.

| Sumber | Dipakai untuk | Tanggal | Status |
|---|---|---|---|
| 7 berkas paket lomba | semua bagian | resmi | boleh |
| SKB 3 Menteri libur nasional dan cuti bersama 2025 ([Bisnis.com](https://kabar24.bisnis.com/read/20241015/79/1807513/sah-ini-daftar-hari-libur-nasional-dan-cuti-bersama-2025)) dan revisi cuti 18 Agustus 2025 ([Kontan](https://nasional.kontan.co.id/news/usai-cuti-bersama-1882025-cek-libur-panjang-terdekat)) | `CUTI` | Okt 2024, Agt 2025 | boleh |
| SKB 3 Menteri 2026 ([setneg.go.id](https://setneg.go.id/baca/index/inilah_skb_3_menteri_libur_nasional_dan_cuti_bersama_2026)) | `CUTI` | ditandatangani 19 Sep 2025 | boleh |
| Wikipedia, *Daftar film Indonesia tahun 2025*, [revisi 27901699](https://id.wikipedia.org/w/index.php?oldid=27901699) | fitur `lokal` | revisi 30 Sep 2025 02:44 UTC | boleh |
| SK Kepala Dinas Pendidikan DKI Jakarta No. 89 Tahun 2025 tentang Kalender Pendidikan 2025/2026 | tanggal libur sekolah dan Idulfitri | PDF dibuat 23 Juni 2025; tautan diarsipkan Wayback 13 Juli 2025 | boleh |
| TMDB, Google Trends, riwayat penonton filmindonesia.or.id | uji coba saja (11.2) | diambil Oktober 2026 (TMDB, Trends); snapshot Wayback sebelum 30 Sep 2025 (filmindonesia) | TMDB dan Trends **tidak boleh**; seluruhnya tidak masuk model akhir |

### 11.2 Eksperimen yang diuji dan gugur
Semua diuji dengan protokol bagian 5 pada pipeline yang sama (rincian dan angka lengkap di notebook riset `EDA_Tiket_Bioskop.ipynb`). Angka berikut selisih terhadap model tanpa fitur itu
(positif = lebih buruk).

| Ide | Hasil |
|---|---|
| Retensi per klaster, kompetisi rilis film lain, tren dan bentuk nasional film, jumlah klaster film | memburuk (CV +0,001 sampai +0,023); dengan 171 film latih model menghafal film |
| Fitur show, okupansi, tiket per show, kapasitas kursi klaster | OOT +0,01 sampai +0,02, atau selisih di dalam noise |
| Popularitas, vote, runtime, anggaran, koleksi (TMDB) | memburuk di CV dan OOT; juga tidak patuh aturan |
| Google Trends (tiga bentuk sinyal) | CVw-2D +0,004 sampai +0,011 |
| Rekam jejak sutradara, produser, sekuel dari 195 film terlaris (filmindonesia.or.id) | CVw-2D +0,001 sampai +0,011 |
| Prior daya tahan dari kru dan pemeran di `movies.csv` | CV +0,013 |
| Indeks pasar harian, tanggal gajian, libur sekolah sebagai fitur | tidak membaik / di dalam noise |
| Kuantil 0,55-0,70 dan loss kuadrat | memburuk (bagian 8) |
| Ambang gerbang nol 0,45 / 0,65 / 0,75 / tanpa gerbang | selisih kecil; 0,55 dipertahankan |
| Model terpisah menurut skala | CVw-2D +0,07 |
| Pembobotan kebaruan, latih bertahap per bulan | tidak ada arah drift yang konsisten; pembobotan kebaruan memburuk |

### 11.3 Batas yang harus diketahui pembaca
1. **Informasi tingkat film tidak tertangkap.** Jika kurva penurunan tiap film test diketahui, skor validasi akan turun ke sekitar 0,40; model ini tidak punya sumber legal untuk itu.
   Sisa galat terbesar ada di sini dan di pasangan berskala kecil pada film rilis lebar.
2. **Koreksi level 1,15 bersandar pada leaderboard publik** (10% data), jadi punya risiko menyesuaikan diri ke subset itu. Koreksi libur sekolah dan Lebaran punya padanan terukur di data latih.
3. **Ramadan, Natal, Tahun Baru, Imlek tidak dikoreksi.** Efek tanggal yang diukur dari `test_history` untuk Natal (+0,27 skala log) dan Tahun Baru (+0,31) mirip libur nasional di train,
   sehingga model sudah menanganinya lewat fitur libur. Imlek dan Nyepi tidak melonjak, tetapi koreksinya (paling banyak 0,002) tidak cukup terbukti.
4. **Koreksi Lebaran mengenai tujuh film** (6,1% baris). Faktor 1,5 adalah batas bawah yang disengaja terhadap indeks pasar Lebaran 2025.
5. **Tidak ada label test**, jadi tidak ada sel di notebook ini yang bisa memverifikasi skor leaderboard.